# 01_config_constant_isotropic

- Constant isotropic covariance: Cov = lambda I_d. Baseline for numerical sanity.
- Config: `configs/config_constant_isotropic.yaml`
- Workflow: simulate -> realized covariance spectrum -> MP inverse -> recovery metrics.


In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'mpdiff').exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root (expected pyproject.toml and src/mpdiff).')
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from mpdiff.config.loader import load_config
from mpdiff.experiments.run_end_to_end import run_end_to_end

config_path = PROJECT_ROOT / 'configs/config_constant_isotropic.yaml'
cfg = load_config(config_path)
out_dir = Path(cfg.global_settings.output_dir)
out_dir


ModuleNotFoundError: No module named 'mpdiff'

In [ ]:
summary = run_end_to_end(config_path)
summary


In [ ]:
summary_csv = out_dir / 'full_pipeline_method_summary.csv'
meta_json = out_dir / 'full_pipeline_metadata.json'
report_txt = out_dir / 'full_pipeline_report.txt'

df = pd.read_csv(summary_csv)
display(df)

meta = json.loads(meta_json.read_text())
print('timers (seconds):')
for k, v in meta['timers_seconds'].items():
    print(f'  {k}: {v:.4f}')

print('\nreport preview:')
print(report_txt.read_text()[:1600])


In [ ]:
fig_paths = [
    out_dir / 'full_pipeline_overlay_population_empirical_recovered.png',
    out_dir / 'full_pipeline_overlay_observed_reconstructed_forward.png',
    out_dir / 'full_pipeline_eigen_histograms.png',
    out_dir / 'full_pipeline_method_population_comparison.png',
]

for p in fig_paths:
    if p.exists():
        img = mpimg.imread(p)
        plt.figure(figsize=(10, 4.5))
        plt.imshow(img)
        plt.title(p.name)
        plt.axis('off')
        plt.show()


## Interpretation Checklist

- Compare `population_wasserstein_1` and `reconstruction_l2` jointly.
- Inspect whether best method is stable across nearby config choices.
- For piecewise models, remember reference law is based on integrated covariance over time.
